In [2]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# 1. 데이터 로드 (경로 그대로 유지)
# =========================
df = pd.read_excel("../데이터 베이스/2. 12개 입력값 머신러닝 전단파괴만.xlsx")

y = df.iloc[:, -1].values          # Pu (마지막 열)
X = df.iloc[:, :-1].copy()

# ID 컬럼 있으면 제거
for col in ["TestID", "ID", "Name"]:
    if col in X.columns:
        X = X.drop(columns=[col])

# (선택) 변수 매핑 확인용: x0, x1, ...가 어떤 컬럼인지 출력
print("=== X column order (x0, x1, ...) ===")
for i, c in enumerate(X.columns):
    print(f"x{i} = {c}")

X = X.values

# =========================
# 2. Train / Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3. PySR 설정
# =========================
model = PySRRegressor(
    niterations=2000,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square"],     # x² 허용
    maxsize=15,                     # 식 복잡도 제한
    elementwise_loss="(x - y)^2",   # 최신 옵션 (loss 경고 방지)
    model_selection="best",
    verbosity=1,
    random_state=42,
    deterministic=True,             # 재현성
    parallelism="serial",           # 재현성
)

# =========================
# 4. 학습
# =========================
model.fit(X_train, y_train)

# =========================
# 5. 예측 성능 (R², RMSE, MAE)
# =========================
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)

# sklearn 구버전 호환: squared=False 대신 sqrt(MSE)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

mae = mean_absolute_error(y_test, y_pred)

print("\n=== Test performance ===")
print("R²  =", r2)
print("RMSE =", rmse)
print("MAE  =", mae)

# =========================
# 6. 최종 경험식
# =========================
print("\n=== Best equation (model_selection='best') ===")
print(model)

# =========================
# 7. 후보식 테이블(복잡도별 Hall of Fame)
# =========================
print("\n=== Hall of Fame equations table ===")
print(model.equations_)


=== X column order (x0, x1, ...) ===
x0 = id
x1 = d
x2 = h
x3 = b_0
x4 = b
x5 = A
x6 = b_h_col
x7 = fc
x8 = fy
x9 = r
x10 = a_d
x11 = fy_fc
x12 = b_d


c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


JuliaError: UndefVarError: `x` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Stacktrace:
 [1] top-level scope
   @ none:1
 [2] eval
   @ .\boot.jl:430 [inlined]
 [3] eval
   @ .\Base.jl:130 [inlined]
 [4] pyjlmodule_seval(self::Module, expr::Py)
   @ PythonCall.JlWrap C:\Users\SSC-3\.julia\packages\PythonCall\avYrV\src\JlWrap\module.jl:13
 [5] _pyjl_callmethod(f::Any, self_::Ptr{PythonCall.C.PyObject}, args_::Ptr{PythonCall.C.PyObject}, nargs::Int64)
   @ PythonCall.JlWrap C:\Users\SSC-3\.julia\packages\PythonCall\avYrV\src\JlWrap\base.jl:67
 [6] _pyjl_callmethod(o::Ptr{PythonCall.C.PyObject}, args::Ptr{PythonCall.C.PyObject})
   @ PythonCall.JlWrap.Cjl C:\Users\SSC-3\.julia\packages\PythonCall\avYrV\src\JlWrap\C.jl:63

In [ ]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# 1) 데이터 로드 (경로 그대로)
# =========================
df = pd.read_excel("../데이터 베이스/2. 12개 입력값 머신러닝 전단파괴만.xlsx")

y = df.iloc[:, -1].astype(float).values          # Vn
Xdf = df.iloc[:, :-1].copy()

# ✅ id 제거 (너 파일은 id가 입력에 들어가 있었음)
for col in ["id", "TestID", "ID", "Name"]:
    if col in Xdf.columns:
        Xdf = Xdf.drop(columns=[col])

# ✅ float 강제
X = Xdf.astype(float).values

# (선택) x0, x1 매핑 확인
print("=== Variable index mapping (x0, x1, ...) ===")
for i, c in enumerate(Xdf.columns):
    print(f"x{i} = {c}")

# =========================
# 2) Train / Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3) PySR 설정
# =========================
model = PySRRegressor(
    niterations=2000,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square"],
    maxsize=15,
    model_selection="best",
    verbosity=1,
    deterministic=True,
    parallelism="serial",
    random_state=42,
    variable_names=list(Xdf.columns),
)

# =========================
# 4) 학습
# =========================
model.fit(X_train, y_train)

# =========================
# 5) 성능 (R² / RMSE / MAE)
# =========================
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # sklearn 호환
mae = mean_absolute_error(y_test, y_pred)

print("\n=== Test performance ===")
print("R²  =", r2)
print("RMSE =", rmse)
print("MAE  =", mae)

# =========================
# 6) 최종 경험식
# =========================
print("\n=== Best equation ===")
print(model)

print("\n=== Hall of Fame ===")
print(model.equations_)


=== Variable index mapping (x0, x1, ...) ===
x0 = d
x1 = h
x2 = b_0
x3 = b
x4 = A
x5 = b_h_col
x6 = fc
x7 = fy
x8 = r
x9 = a_d
x10 = fy_fc
x11 = b_d


c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
